# Iterative Intelligence Playbook: The Path to Benchmark-Beating RL

This notebook illustrates the systematic, data-driven process we have followed to optimize the SAC trading agent. Each phase represents a strategic "pivot" based on experiment results and AI-led analysis.

## 🛠️ Phase 1: Solving the Inactivity Collapse

Initially, the SAC agent suffered from "inactivity collapse," where it would simply stay in cash (0 shares) to avoid transaction costs.

**The Pivot:**
- **Entropy Tuning (`ent_coef`)**: Decreased from `auto` to `0.01` to encourage more stable but directional exploration.
- **Action Bonus**: Introduced a nudge to reward positioning.

## 📊 Phase 2: Stationarity & News Integration

**The Pivot:**
- **Stationary Features**: Switched to log returns, relative ranges, and normalized volumes.
- **News Sentiment**: Integrated Gemini-processed news sentiment.

In [ ]:
# Fix pathing for notebook-level imports
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Example: Generating the optimized training frame
from src.market_data import get_tech_training_data

df = get_tech_training_data(
    include_news=True, 
    use_stationary_features=True
)
print(f"Training columns: {df.columns.tolist() if 'df' in locals() else 'Run this cell to see'}")

## 📈 Phase 3: Advanced Technical Indicator Expansion

**The Expansion:**
- **ATR, Bollinger Bands, SMA Crossovers**
- **VWAP (Relative)**: Connects price action to volume distribution.
- **MACD Signal/Histogram**: Captures momentum acceleration.

## ⚠️ Handling the "Observation Space Mismatch"

Expanding indicators is a double-edged sword. When we add columns (like VWAP), the **Environment Shape** changes.

**The Problem:**
- Older models (trained on 15 features) will fail when loaded into an environment now providing 25+ features.

**The Fix (Alignment):**
1. **Refresh Parquet Cache**: Re-run the data fetcher with `refresh=True` to write the new columns to disk.
2. **Force-Train New Model**: Any new sweep will automatically pick up the newest `potential_cols` definition.

## 🤖 Phase 4: AI Strategic Analyst Integration

**The Workflow:**
1. **Experiment** -> Generates `experiment_leaderboard.csv`
2. **Quant Report** -> Renders markdown stats.
3. **AI Critique** -> Provides a "Confidence Score" and "Strategic Pivot" recommendation.

## 🔄 The Iterative Tuning Loop

Tuning an RL agent is a **continuous feedback loop** between the researcher and the AI Analyst.

1. **Hypothesis** -> "Higher Entropy will improve test generalization."
2. **The Sweep** -> Multi-seeded experiment.
3. **Statistical Diagnostic** -> `quant_report.py` (Math check).
4. **The LLM Critique** -> Identify "Personality" flaws (Inactivity, Overfit).
5. **Strategic Pivot** -> Adjust knobs (Reward Scales, Entropy, Steps) and re-run.

## 🚀 Current Frontier: The Goldilocks Zone

**Current Hyperparameters (`pressure-v1` / `hybrid-v1`):**
- `ent_coef`: 0.02-0.05 (Balanced exploration)
- `reward_direction_scale`: 1.0 (Heavy weight on being right)
- `reward_action_bonus_scale`: 0.25 (Aggressive nudge to trade)
- `reward_mode`: `sortino` (Focusing on downside risk-adjusted return)

---